### 1: Importações e Carga dos Dados (Série Contínua)

In [0]:
%pip install statsmodels

In [0]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pyspark.sql import functions as F

# 1. Ler a tabela Gold com a série temporal horária
df_ts = spark.table("workspace.default.gold_series_temporais_pedidos_hora")

# Converter para Pandas e preparar o índice temporal contínuo
pdf_ts = df_ts.orderBy("data_dia", "hora").toPandas()
pdf_ts["timestamp"] = pd.to_datetime(pdf_ts["data_dia"]) + pd.to_timedelta(pdf_ts["hora"], unit="h")
pdf_ts = pdf_ts.set_index("timestamp").sort_index()

# Recortar no último trecho contínuo de dados reais (remove gap final >1 dia, ex.: ~20 dias de horas zeradas)
gap_inicio = pdf_ts.index.to_series().diff() > pd.Timedelta(days=1)
gap_inicio = gap_inicio.reset_index(drop=True)
if gap_inicio.any():
    pdf_ts = pdf_ts.iloc[: gap_inicio[::-1].idxmax()]

# Preencher horas faltantes na série e criar variáveis exógenas
pdf_ts = pdf_ts.asfreq("h", fill_value=0)
pdf_ts["hora"] = pdf_ts.index.hour
pdf_ts["dia_da_semana"] = pdf_ts.index.dayofweek + 1

print(f"Total de registros na série horária contínua: {len(pdf_ts)}")
display(pdf_ts.head(5))

### 2: Divisão de Treino e Teste (Foco na Tarde: 12h - 18h)

In [0]:
# Separar treino (histórico) e teste (último dia COM dados reais à tarde; data_dia==0 são horas preenchidas)
ultimo_dia = pdf_ts[(pdf_ts["hora"].between(12, 18)) & (pdf_ts["data_dia"] != 0)].index.normalize().max()
treino = pdf_ts[pdf_ts.index.normalize() < ultimo_dia]
teste = pdf_ts[pdf_ts.index.normalize() == ultimo_dia]

# Filtrar o teste estritamente para a faixa da tarde (12h às 18h)
teste_tarde = teste[(teste["hora"] >= 12) & (teste["hora"] <= 18)]

print(f"Período de Treino: até {ultimo_dia.strftime('%Y-%m-%d')}")
print(f"Período de Teste (Tarde 12h-18h): {len(teste_tarde)} horas")
display(teste_tarde)

### 3: Treinamento e Previsão do SARIMAX — Receita

In [0]:
# Definir variáveis exógenas
exog_treino = treino[["hora", "dia_da_semana"]]
exog_teste = teste_tarde[["hora", "dia_da_semana"]]

# Treinar o modelo SARIMAX para RECEITA
modelo_receita = SARIMAX(
    treino["receita"], 
    exog=exog_treino,
    order=(1, 1, 1), 
    seasonal_order=(1, 0, 0, 24),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

# Previsão para a tarde
pred_receita = modelo_receita.forecast(steps=len(teste_tarde), exog=exog_teste)
print("Modelo SARIMAX de Receita treinado e previsto com sucesso!")

### 4: Treinamento e Previsão do SARIMAX — Número de Pedidos

In [0]:
# Treinar o modelo SARIMAX para NÚMERO DE PEDIDOS
modelo_pedidos = SARIMAX(
    treino["num_pedidos"], 
    exog=exog_treino,
    order=(1, 1, 1), 
    seasonal_order=(1, 0, 0, 24),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

# Previsão para a tarde
pred_pedidos = modelo_pedidos.forecast(steps=len(teste_tarde), exog=exog_teste)
print("Modelo SARIMAX de Número de Pedidos treinado e previsto com sucesso!")

### 5: Avaliação e Cálculo das Métricas (MAE, WAPE, RMSE)

In [0]:
# --- Cálculo para RECEITA ---
y_true_rec = teste_tarde["receita"].values
y_pred_rec = pred_receita.values
mae_rec = np.mean(np.abs(y_true_rec - y_pred_rec))
rmse_rec = np.sqrt(np.mean((y_true_rec - y_pred_rec) ** 2))
wape_rec = np.sum(np.abs(y_true_rec - y_pred_rec)) / np.sum(y_true_rec)

# --- Cálculo para NÚMERO DE PEDIDOS ---
y_true_ped = teste_tarde["num_pedidos"].values
y_pred_ped = pred_pedidos.values
mae_ped = np.mean(np.abs(y_true_ped - y_pred_ped))
rmse_ped = np.sqrt(np.mean((y_true_ped - y_pred_ped) ** 2))
wape_ped = np.sum(np.abs(y_true_ped - y_pred_ped)) / np.sum(y_true_ped)

print("================ MÉTRICAS DO SARIMAX (12h - 18h) ================")
print(f"--- RECEITA ---")
print(f"MAE:  R$ {mae_rec:.2f}")
print(f"RMSE: R$ {rmse_rec:.2f}")
print(f"WAPE: {wape_rec * 100:.2f}%")

print(f"\n--- NÚMERO DE PEDIDOS ---")
print(f"MAE:  {mae_ped:.2f} pedidos")
print(f"RMSE: {rmse_ped:.2f} pedidos")
print(f"WAPE: {wape_ped * 100:.2f}%")
print("==================================================================")


###: Criar a Tabela Final para o Looker (No Databricks)

In [0]:
# Unir os dados reais, do baseline e as previsões do SARIMAX em um único DataFrame Pandas para salvar no Databricks
df_export = teste_tarde.copy()
df_export["receita_sarimax"] = pred_receita.values
df_export["pedidos_sarimax"] = pred_pedidos.values

# Converter de volta para Spark DataFrame
spark_df_export = spark.createDataFrame(df_export.reset_index())

# Salvar a tabela Gold pronta para o Looker
spark_df_export.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_dashboard_sarimax_tarde")

print("Tabela 'gold_dashboard_sarimax_tarde' salva com sucesso para o Looker!")

In [0]:
# Recriar a tabela gold garantindo todas as colunas para o Looker
df_export = teste_tarde.copy()
df_export["receita_sarimax"] = pred_receita.values
df_export["pedidos_sarimax"] = pred_pedidos.values

# Garantir que as colunas de baseline existam no dataframe exportado
# (se já existiam, mantemos; se não, calculamos o lag equivalente)
if "receita_baseline" not in df_export.columns:
    df_export["receita_baseline"] = df_export["receita"].shift(1) # Ajuste simples se necessário
if "pedidos_baseline" not in df_export.columns:
    df_export["pedidos_baseline"] = df_export["num_pedidos"].shift(1)

# Renomear para alinhar com a query do Looker
df_export = df_export.rename(columns={
    "receita": "receita_real",
    "num_pedidos": "pedidos_real"
})

# Converter de volta para Spark DataFrame e salvar
spark_df_export = spark.createDataFrame(df_export.reset_index())
spark_df_export.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_dashboard_sarimax_tarde")

print("Tabela atualizada e corrigida com sucesso para o Looker!")